In [2]:
!pip install transformers datasets sentencepiece accelerate evaluate sacrebleu -q

In [3]:
# Load Dataset
from datasets import load_dataset

dataset = load_dataset("SPEAK-PP/sinhala-combined-clean-dyslexic-itn")
print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/333 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.71M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45847 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dyslexic_sentence', 'clean_sentence'],
        num_rows: 45847
    })
})
{'dyslexic_sentence': 'පස්සේ ඔවුන් උස්සාගෙන ගියා.මම මේ වත්තේ සිට තැනින් තැන වතුවලට මාරුවෙමින් වසර අටක පමණ සේවය කරලා තියෙනවා.', 'clean_sentence': 'පස්සේ ඔවුන් උස්සාගෙන ගියා.මම මේ වත්තේ සිට තැනින් තැන වතුවලට මාරුවෙමින් වසර 8ක පමණ සේවය කරලා තියෙනවා.'}


In [4]:
# Load mt-5 model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
# Preprocessing
# Check column names first
print(dataset["train"].column_names)

['dyslexic_sentence', 'clean_sentence']


In [6]:
INPUT_COL = "dyslexic_sentence"
OUTPUT_COL = "clean_sentence"

PREFIX = "normalize sinhala: "
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 128

def preprocess(examples):
    inputs = [PREFIX + str(text) for text in examples[INPUT_COL]]
    targets = [str(text) for text in examples[OUTPUT_COL]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )

    # Replace padding token id with -100 so it's ignored in loss
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(preprocess, batched=True)
print(tokenized)

Map:   0%|          | 0/45847 [00:00<?, ? examples/s]

Error during conversion: ReadTimeout('The read operation timed out')


model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['dyslexic_sentence', 'clean_sentence', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 45847
    })
})


In [7]:
# Training Arguments
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-sinhala-itn",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=10,
    num_train_epochs=3,         # Total number or training epochs
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=True,                      # Enable for GPU speedup
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [8]:
# Define Metrics
import evaluate
import numpy as np

bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]

    result = bleu.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": round(result["score"], 2)}

In [9]:
# Split train into train + validation (90/10)
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

split = tokenized["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,0.000000,nan,0.080000
2,0.000000,nan,0.080000
3,0.000000,nan,0.080000
4,0.000000,nan,0.080000
5,0.000000,nan,0.080000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12895, training_loss=0.0, metrics={'train_runtime': 2880.322, 'train_samples_per_second': 71.627, 'train_steps_per_second': 4.477, 'total_flos': 6.184380185837568e+16, 'train_loss': 0.0, 'epoch': 5.0})

In [11]:
model.save_pretrained("./mt5-si-itn")
tokenizer.save_pretrained("./mt5-si-itn")
print("Model Saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved.


In [15]:
input_text = "normalize sinhala: න්ක ඔන් එකෙතියෙන්නේ කපල නෑ පියෝටීවිසහ ඔයිස් දෙක මෝන්න කි තියේන සමොකද්වෙලා තියෙන්නේ වැඩකර පියෝටීවිකත් වැඩකරනා ටවඉස් එකත් හරිමේ ලයින් එකේ ඒ පහසු කන්දක මෝන් කෙසර තියෙනන ගැටලුවක් නෑ ලයින් රනාඩෙදාස් පන්සේ අසුතුනත්ත සර්ගෙවන්නතියනැච්චර ගවන්න තියෙන්නේවසසෙ පොඩ්ඩක් බලන්න කියන  මේ සමහරක් ගලට මේ කේබල් වහිනකොට ගලවනානේ අරකළුපාඩ බොක්සකින් නවා නිට් ගේලක තියෙනන හතරෙන්නේ කවුලලිව පොඩ්ඩක් හමට ගැලිල ඇති පශික්කරනකන් මේ පත්තනං ඔව කෙස හරිසහරිවකස එසටමත සවදස් සබදාසක් ස ඇමතිමකිසනා රඳඉන්න"

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_length=128)
print(tokenizer.decode(output[0], skip_special_tokens=True))

<extra_id_0> sinhala - sinhala - -


In [14]:
# Inference
from transformers import pipeline

pipe = pipeline(
    "text2text-generation",
    model="./mt5-si-itn",
    tokenizer=tokenizer,
    device=0  # use GPU
)

test_input = "normalize sinhala: න්ක ඔන් එකෙතියෙන්නේ කපල නෑ පියෝටීවිසහ ඔයිස් දෙක මෝන්න කි තියේන සමොකද්වෙලා තියෙන්නේ වැඩකර පියෝටීවිකත් වැඩකරනා ටවඉස් එකත් හරිමේ ලයින් එකේ ඒ පහසු කන්දක මෝන් කෙසර තියෙනන ගැටලුවක් නෑ ලයින් රනාඩෙදාස් පන්සේ අසුතුනත්ත සර්ගෙවන්නතියනැච්චර ගවන්න තියෙන්නේවසසෙ පොඩ්ඩක් බලන්න කියන  මේ සමහරක් ගලට මේ කේබල් වහිනකොට ගලවනානේ අරකළුපාඩ බොක්සකින් නවා නිට් ගේලක තියෙනන හතරෙන්නේ කවුලලිව පොඩ්ඩක් හමට ගැලිල ඇති පශික්කරනකන් මේ පත්තනං ඔව කෙස හරිසහරිවකස එසටමත සවදස් සබදාසක් ස ඇමතිමකිසනා රඳඉන්න"  # example input
result = pipe(test_input, max_length=128)
print("Output:", result[0]["generated_text"])

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"